## Muestreos

## Muestreo por conglomerado

In [1]:
import pandas as pd
import random

# 1. Simulación de la Base de Datos
# Creamos 50 centros de salud, cada uno con un número variable de historias clínicas (entre 50 y 100)
random.seed(42)  # Para reproductibilidad
centros_salud = []
for i in range(1, 51):
    n_historias = random.randint(50, 100)
    for j in range(1, n_historias + 1):
        centros_salud.append({
            'ID_Centro': f'Centro_{i}',
            'ID_Historia': f'HC_{i}_{j}',
            'Calidad_Atencion': random.randint(0, 100)  # Escala 0-100
        })

df_provincia = pd.DataFrame(centros_salud)

# --- INICIO DEL MUESTREO BIETÁPICO ---

# ETAPA 1: Seleccionar 5 centros de salud (conglomerados)
todos_los_centros = df_provincia['ID_Centro'].unique()
centros_seleccionados = random.sample(list(todos_los_centros), k=5)

print(f"ETAPA 1: Centros seleccionados: {centros_seleccionados}\n")

# ETAPA 2: Dentro de cada centro seleccionado, auditar 10 historias clínicas
muestra_final = []

for centro in centros_seleccionados:
    # Filtramos la base de datos por el centro actual
    universo_centro = df_provincia[df_provincia['ID_Centro'] == centro]
    
    # Seleccionamos 10 historias al azar de ese centro específico
    muestra_centro = universo_centro.sample(n=10)
    muestra_final.append(muestra_centro)

# Consolidamos la muestra total
df_muestra_final = pd.concat(muestra_final)

# --- RESULTADOS ---

print("RESULTADOS DE LA AUDITORÍA (Muestra de 50 registros):")
print(df_muestra_final.head(15)) # Mostramos los primeros 15 registros de la muestra

# Análisis rápido de la muestra
print("\nResumen Estadístico de la Calidad de Atención en la Muestra:")
print(df_muestra_final.groupby('ID_Centro')['Calidad_Atencion'].mean())
print(f"\nPromedio General de la Provincia (estimado): {df_muestra_final['Calidad_Atencion'].mean():.2f}")

ETAPA 1: Centros seleccionados: ['Centro_35', 'Centro_49', 'Centro_3', 'Centro_47', 'Centro_29']

RESULTADOS DE LA AUDITORÍA (Muestra de 50 registros):
      ID_Centro ID_Historia  Calidad_Atencion
2664  Centro_35    HC_35_39                17
2639  Centro_35    HC_35_14                57
2695  Centro_35    HC_35_70                86
2691  Centro_35    HC_35_66                60
2633  Centro_35     HC_35_8                46
2690  Centro_35    HC_35_65                38
2689  Centro_35    HC_35_64                72
2676  Centro_35    HC_35_51                40
2700  Centro_35    HC_35_75                92
2631  Centro_35     HC_35_6                71
3713  Centro_49    HC_49_14                66
3743  Centro_49    HC_49_44                96
3738  Centro_49    HC_49_39                 0
3749  Centro_49    HC_49_50                23
3704  Centro_49     HC_49_5                14

Resumen Estadístico de la Calidad de Atención en la Muestra:
ID_Centro
Centro_29    41.9
Centro_3     64.1
Cent

In [1]:
import math
from scipy import stats

def calcular_muestra_finita(N, sigma, d, confianza=0.95):
    """
    Calcula el tamaño de muestra para una media en población finita.
    """
    # 1. Obtención del valor crítico Z (bilateral)
    alfa = 1 - confianza
    z = stats.norm.ppf(1 - alfa/2)
    
    # 2. Cálculo para población infinita (n0)
    n0 = (z**2 * sigma**2) / (d**2)
    
    # 3. Ajuste para población finita (n)
    n_final = n0 / (1 + (n0 / N))
    
    # 4. Redondeo al entero superior inmediato (Regla de oro en bioestadística)
    # Es necesario aumentar al siguiente entero para no perder potencia [6, 15, 19].
    return math.ceil(n_final), n0

# Parámetros del problema
N_pacientes = 800
desviacion_estandar = 1.5
precision_d = 0.2
nivel_confianza = 0.95

n_ajustado, n_inf = calcular_muestra_finita(N_pacientes, desviacion_estandar, precision_d, nivel_confianza)

print(f"--- Resultados del Cálculo Muestral ---")
print(f"Tamaño base (población infinita): {math.ceil(n_inf)}")
print(f"Tamaño ajustado (cohorte N=800): {n_ajustado} pacientes")

--- Resultados del Cálculo Muestral ---
Tamaño base (población infinita): 217
Tamaño ajustado (cohorte N=800): 171 pacientes
